# 01 - Data Cleaning

Goal: Clean the OpenAQ measurements data and prepare for analysis.

## Steps
1. Load raw measurements from CSV
2. Inspect data quality (real-world data has gaps and quirks)
3. Pivot to wide format (one row per city/date, columns per pollutant)
4. Handle missing values appropriately
5. Add time features
6. Compute simplified AQI proxy
7. Save processed data

In [33]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

## 1. Load Raw Data

In [34]:
raw_files = sorted(Path('../data/raw').glob('openaq_measurements_*.csv'))
if not raw_files:
    raise FileNotFoundError('No measurements CSV found. Run scripts/fetch_measurements.py first.')

latest = raw_files[-1]
df = pd.read_csv(latest)
print(f'Loaded: {latest.name}')
print(f'Shape: {df.shape}')
df.head()

Loaded: openaq_measurements_20260526.csv
Shape: (68944, 11)


,country,city,location_name,location_id,sensor_id,parameter,units,lat,lon,datetime,value
0,Indonesia,NaN,Jakarta South,8320,5077558,pm25,µg/m³,-6.236704,106.793244,2022-10-04T00:00:00+07:00,27.7
1,Indonesia,NaN,Jakarta South,8320,5077558,pm25,µg/m³,-6.236704,106.793244,2022-10-05T00:00:00+07:00,39.0
2,Indonesia,NaN,Jakarta South,8320,5077558,pm25,µg/m³,-6.236704,106.793244,2023-05-07T00:00:00+07:00,NaN
3,Indonesia,NaN,Jakarta South,8320,5077558,pm25,µg/m³,-6.236704,106.793244,2023-05-08T00:00:00+07:00,NaN
4,Indonesia,NaN,Jakarta South,8320,5077558,pm25,µg/m³,-6.236704,106.793244,2023-05-09T00:00:00+07:00,45.9


### Fix City Names

The fetched data has missing city names because OpenAQ's `locality` field is often empty.
We re-attach city info from the locations CSV (which used GPS coords to assign cities).

In [35]:
# Load locations CSV (has GPS-derived cities)
locations = pd.read_csv('../data/raw/asean_locations.csv')
city_map = dict(zip(locations['id'], locations['locality']))

# Override the messy 'city' column using the location_id mapping
df['city'] = df['location_id'].map(city_map).fillna(df['city']).fillna('Unknown')

# Clean weird entries
df['city'] = df['city'].astype(str).str.strip()
df.loc[df['city'].isin(['nan', 'None', '']), 'city'] = 'Unknown'

print('City distribution after fix:')
print(df.groupby(['country', 'city']).size().sort_values(ascending=False).head(20))

City distribution after fix:
country      city                              
Thailand     Bangkok                               10936
Vietnam      Hanoi                                 10469
Thailand     Chachoengsao                           4639
Indonesia    Jakarta                                4459
Thailand     Amnat Charoen                          4153
             Bueng Kan                              4132
Indonesia    Unknown                                3101
Singapore    Singapore                              2523
Vietnam      Ho Chi Minh City                       1815
Myanmar      Unknown                                1339
Thailand     Chai Nat                               1180
Philippines  Manila                                 1040
Malaysia     Kuala Lumpur                           1002
Cambodia     AUPP Liger Leadership Academy           947
Philippines  Iloilo City, Augustine Grove Subd.      832
Indonesia    Depok                                   782
Malaysia   

### Re-classify cities using stricter GPS bounding boxes

Some sensors were mis-labeled (e.g., a sensor in Depok showing as Jakarta).
Let's use stricter coordinate-based city assignment for accuracy.

In [36]:
# Stricter city bounding boxes (lat_min, lat_max, lon_min, lon_max)
STRICT_CITY_BOUNDS = {
    'Indonesia': [
        # Jakarta proper (excluding Depok, Tangerang, Bekasi)
        ('Jakarta',    -6.30, -6.05, 106.70, 106.97),
        ('Depok',      -6.45, -6.30, 106.75, 106.90),
        ('Tangerang',  -6.30, -6.10, 106.50, 106.70),
        ('Bekasi',     -6.30, -6.15, 106.95, 107.10),
        ('Bogor',      -6.65, -6.45, 106.70, 106.85),
        ('Bandung',    -7.05, -6.80, 107.50, 107.75),
        ('Surabaya',   -7.40, -7.15, 112.60, 112.85),
        ('Yogyakarta', -7.95, -7.60, 110.30, 110.55),
        ('Semarang',   -7.05, -6.85, 110.30, 110.50),
        ('Medan',       3.45,  3.75,  98.55,  98.80),
        ('Palembang',  -3.10, -2.85, 104.65, 104.90),
        ('Padang',     -1.05, -0.50, 100.30, 100.50),
        ('Denpasar',   -8.80, -8.55, 115.10, 115.35),
        ('Makassar',   -5.25, -5.05, 119.35, 119.55),
    ],
    'Thailand': [
        ('Bangkok',     13.55, 13.95, 100.40, 100.80),
        ('Chiang Mai',  18.65, 18.85,  98.90,  99.05),
        ('Phuket',       7.85,  8.05,  98.30,  98.45),
    ],
    'Vietnam': [
        ('Ho Chi Minh City', 10.65, 10.90, 106.55, 106.85),
        ('Hanoi',            20.95, 21.15, 105.70, 105.95),
    ],
    'Philippines': [
        ('Manila',      14.50, 14.75, 120.95, 121.10),
        ('Quezon City', 14.60, 14.80, 121.00, 121.15),
        ('Makati',      14.50, 14.60, 121.00, 121.05),
    ],
    'Singapore': [
        ('Singapore',    1.15,  1.50, 103.55, 104.05),
    ],
    'Malaysia': [
        ('Kuala Lumpur', 3.05,  3.25, 101.60, 101.80),
    ],
    'Cambodia': [
        ('Phnom Penh',  11.50, 11.65, 104.85, 105.00),
    ],
    'Myanmar': [
        ('Yangon',      16.75, 16.95,  96.05,  96.25),
    ],
}

def assign_city_strict(country, lat, lon):
    """Assign city based on strict GPS bounding boxes."""
    if pd.isna(lat) or pd.isna(lon):
        return 'Unknown'
    bounds = STRICT_CITY_BOUNDS.get(country, [])
    for city, lat_min, lat_max, lon_min, lon_max in bounds:
        if lat_min <= lat <= lat_max and lon_min <= lon <= lon_max:
            return city
    return 'Other'

# Apply re-classification
df['city'] = df.apply(lambda r: assign_city_strict(r['country'], r['lat'], r['lon']), axis=1)

# Drop sensors that ended up in 'Other' or have wildly wrong coordinates
# (e.g., the 'Jakarta' sensor at lat=-0.6, lon=100 — that's 1000km away in Sumatra)
before = len(df)
df = df[df['city'] != 'Other']
print(f'Removed {before - len(df)} rows with sensors outside known city boundaries')

# Re-check distribution
print('\nCity distribution after strict GPS classification:')
print(df.groupby(['country', 'city']).size().sort_values(ascending=False).head(20))

# Show sensor counts per city
print('\nSensors per city (Indonesia):')
print(df[df['country'] == 'Indonesia'].groupby('city')['location_id'].nunique().sort_values(ascending=False))

Removed 29850 rows with sensors outside known city boundaries

City distribution after strict GPS classification:
country      city            
Thailand     Bangkok             10936
Vietnam      Hanoi               10469
Indonesia    Jakarta              5591
Singapore    Singapore            2523
Indonesia    Depok                1884
Vietnam      Ho Chi Minh City     1815
Myanmar      Yangon               1568
Malaysia     Kuala Lumpur         1518
Philippines  Manila               1040
             Quezon City           682
Indonesia    Yogyakarta            345
             Bogor                 186
             Palembang             176
             Medan                 146
Cambodia     Phnom Penh            111
Indonesia    Bandung               104
dtype: int64

Sensors per city (Indonesia):
city
Jakarta       8
Depok         5
Palembang     3
Yogyakarta    2
Bandung       1
Bogor         1
Medan         1
Name: location_id, dtype: int64


## 2. Inspect Data Quality

In [37]:
print('Column types:')
print(df.dtypes)
print(f'\nMissing values:')
print(df.isna().sum())
print(f'\nDate range: {df["datetime"].min()} to {df["datetime"].max()}')
print(f'\nCountries: {df["country"].unique()}')
print(f'\nPollutants: {df["parameter"].unique()}')
print(f'\nValue summary:')
print(df['value'].describe())

Column types:
country              str
city                 str
location_name        str
location_id        int64
sensor_id          int64
parameter            str
units                str
lat              float64
lon              float64
datetime             str
value            float64
dtype: object

Missing values:
country             0
city                0
location_name       0
location_id         0
sensor_id           0
parameter           0
units               0
lat                 0
lon                 0
datetime            0
value            1689
dtype: int64

Date range: 2021-05-27T00:00:00+07:00 to 2026-05-26T00:00:00+08:00

Countries: <ArrowStringArray>
['Indonesia', 'Singapore', 'Malaysia', 'Thailand', 'Vietnam', 'Philippines', 'Cambodia', 'Myanmar']
Length: 8, dtype: str

Pollutants: <ArrowStringArray>
['pm25', 'o3', 'pm10', 'co', 'no2', 'so2']
Length: 6, dtype: str

Value summary:
count    37405.000000
mean       108.475772
std        641.646919
min          0.000000
25%

In [38]:
# Records by country and pollutant
print('Records per country:')
print(df.groupby('country').size().sort_values(ascending=False))
print('\nRecords per pollutant:')
print(df.groupby('parameter').size().sort_values(ascending=False))

Records per country:
country
Vietnam        12284
Thailand       10936
Indonesia       8432
Singapore       2523
Philippines     1722
Myanmar         1568
Malaysia        1518
Cambodia         111
dtype: int64

Records per pollutant:
parameter
pm25    25312
pm10     5277
o3       3160
no2      2116
co       1723
so2      1506
dtype: int64


## 3. Clean and Parse

In [39]:
# Parse datetime
df['datetime'] = pd.to_datetime(df['datetime'], errors='coerce', utc=True)
df = df.dropna(subset=['datetime'])

# Get date only (we already requested daily aggregates)
df['date'] = df['datetime'].dt.date
df['date'] = pd.to_datetime(df['date'])

# Drop invalid values
before = len(df)
df = df[df['value'].notna() & (df['value'] >= 0)]
print(f'Removed {before - len(df)} rows with invalid values')

Removed 1689 rows with invalid values


In [40]:
# Detect and remove sensor errors (extreme outliers)
# Pollutant-specific reasonable upper bounds (in their typical units)
# NOTE: PM2.5 cap reduced from 1000 to 500 after investigation revealed
# the US Diplomatic Post HCMC sensor was reporting impossible values
# (max 985, sustained averages > 200) in 2022-2023.
# Cross-reference with UN/WHO data: real Vietnam PM2.5 ~30-50 ug/m3.
REASONABLE_LIMITS = {
    'pm25': 500,    # ug/m3 (was 1000 - tightened after sensor investigation)
    'pm10': 2000,   # ug/m3
    'no2': 1000,    # ug/m3
    'so2': 2000,    # ug/m3
    'o3': 1000,     # ug/m3
    'co': 100,      # mg/m3
}

before = len(df)
for pollutant, limit in REASONABLE_LIMITS.items():
    mask = (df['parameter'] == pollutant) & (df['value'] > limit)
    if mask.sum() > 0:
        print(f'  {pollutant}: removing {mask.sum()} extreme readings above {limit}')
        df = df[~mask]
print(f'\nTotal removed: {before - len(df)} rows')

  pm25: removing 149 extreme readings above 500
  co: removing 1514 extreme readings above 100

Total removed: 1663 rows


## 4. Pivot to Wide Format

Currently each row is one (city, date, pollutant). We want one row per (city, date) with columns for each pollutant.

In [41]:
# Aggregate: average pollutant value per city per date (multiple sensors per location)
daily = df.groupby(['country', 'city', 'date', 'parameter'])['value'].mean().reset_index()

# Pivot to wide
wide = daily.pivot_table(
    index=['country', 'city', 'date'],
    columns='parameter',
    values='value',
).reset_index()

wide.columns.name = None
print(f'Wide format shape: {wide.shape}')
wide.head()

Wide format shape: (10430, 9)


,country,city,date,co,no2,o3,pm10,pm25,so2
0,Cambodia,Phnom Penh,2025-05-03,NaN,NaN,NaN,NaN,21.0,NaN
1,Cambodia,Phnom Penh,2025-06-06,NaN,NaN,NaN,NaN,19.2,NaN
2,Cambodia,Phnom Penh,2025-08-14,NaN,NaN,NaN,NaN,21.1,NaN
3,Cambodia,Phnom Penh,2025-09-11,NaN,NaN,NaN,NaN,21.0,NaN
4,Cambodia,Phnom Penh,2025-09-12,NaN,NaN,NaN,NaN,25.1,NaN


## 5. Add Time Features

In [42]:
wide['year'] = wide['date'].dt.year
wide['month'] = wide['date'].dt.month
wide['day_of_week'] = wide['date'].dt.day_name()
wide['quarter'] = wide['date'].dt.quarter

def get_season_sea(month):
    """Southeast Asian seasons: dry vs wet."""
    if month in [12, 1, 2, 3]:
        return 'Dry (Dec-Mar)'
    elif month in [4, 5]:
        return 'Transition (Apr-May)'
    elif month in [6, 7, 8, 9]:
        return 'Wet (Jun-Sep)'
    else:
        return 'Transition (Oct-Nov)'

wide['season'] = wide['month'].apply(get_season_sea)

# COVID period
def covid_period(d):
    if d < pd.Timestamp('2020-03-01'):
        return 'Pre-COVID'
    elif d <= pd.Timestamp('2021-12-31'):
        return 'COVID Era'
    else:
        return 'Post-COVID'

wide['covid_period'] = wide['date'].apply(covid_period)

wide.head()

,country,city,date,co,no2,o3,pm10,pm25,so2,year,month,day_of_week,quarter,season,covid_period
0,Cambodia,Phnom Penh,2025-05-03,NaN,NaN,NaN,NaN,21.0,NaN,2025,5,Saturday,2,Transition (Apr-May),Post-COVID
1,Cambodia,Phnom Penh,2025-06-06,NaN,NaN,NaN,NaN,19.2,NaN,2025,6,Friday,2,Wet (Jun-Sep),Post-COVID
2,Cambodia,Phnom Penh,2025-08-14,NaN,NaN,NaN,NaN,21.1,NaN,2025,8,Thursday,3,Wet (Jun-Sep),Post-COVID
3,Cambodia,Phnom Penh,2025-09-11,NaN,NaN,NaN,NaN,21.0,NaN,2025,9,Thursday,3,Wet (Jun-Sep),Post-COVID
4,Cambodia,Phnom Penh,2025-09-12,NaN,NaN,NaN,NaN,25.1,NaN,2025,9,Friday,3,Wet (Jun-Sep),Post-COVID


## 6. AQI Proxy from PM2.5

OpenAQ doesn't include AQI directly. We compute a proxy using US EPA AQI calculation for PM2.5.

In [43]:
def pm25_to_aqi(pm25):
    """Convert PM2.5 (ug/m3) to US EPA AQI."""
    if pd.isna(pm25):
        return np.nan

    breakpoints = [
        (0.0, 12.0, 0, 50),
        (12.1, 35.4, 51, 100),
        (35.5, 55.4, 101, 150),
        (55.5, 150.4, 151, 200),
        (150.5, 250.4, 201, 300),
        (250.5, 500.4, 301, 500),
    ]

    for c_low, c_high, i_low, i_high in breakpoints:
        if c_low <= pm25 <= c_high:
            return ((i_high - i_low) / (c_high - c_low)) * (pm25 - c_low) + i_low
    return 500  # Cap

def aqi_bucket(aqi):
    if pd.isna(aqi):
        return None
    if aqi <= 50: return 'Good'
    elif aqi <= 100: return 'Moderate'
    elif aqi <= 150: return 'Unhealthy for Sensitive'
    elif aqi <= 200: return 'Unhealthy'
    elif aqi <= 300: return 'Very Unhealthy'
    else: return 'Hazardous'

if 'pm25' in wide.columns:
    wide['aqi_pm25'] = wide['pm25'].apply(pm25_to_aqi)
    wide['aqi_bucket'] = wide['aqi_pm25'].apply(aqi_bucket)
    print(wide['aqi_bucket'].value_counts())

aqi_bucket
Moderate                   5617
Unhealthy for Sensitive    2091
Good                       1599
Unhealthy                  1068
Hazardous                    36
Very Unhealthy               16
Name: count, dtype: int64


## 7. Save Processed Data

In [44]:
out_path = Path('../data/processed/openaq_clean.csv')
out_path.parent.mkdir(parents=True, exist_ok=True)
wide.to_csv(out_path, index=False, encoding='utf-8-sig')
print(f'Saved {len(wide):,} rows to {out_path}')

# Also save as parquet for faster loading
wide.to_parquet('../data/processed/openaq_clean.parquet', index=False)

Saved 10,430 rows to ..\data\processed\openaq_clean.csv
